# 03 · 数据准备：清洗、去重与长度体检

微调效果的上限往往由数据决定。本教程**不需要 GPU**，可完整运行。

## 1. 常见问题

| 问题 | 后果 | 处理 |
|---|---|---|
| 空值 / 缺字段 | 训练报错或学到空输出 | 直接剔除 |
| 近似重复样本 | 过拟合、指标虚高 | 精确 + 模糊去重 |
| 长度极端 | 截断丢信息 / 浪费算力 | 按分位数过滤 |
| 全角半角混杂 | 词表碎片化 | NFKC 归一化 |

**注意**：`DataCleaner` 只检查 `text_field` 指定的那一个字段。
指令数据请传入 `text_field="output"`（或先拼成一个 text 字段）。

In [ ]:
import sys

sys.path.insert(0, "..")

from llm_finetuning import DataCleaner, DataConfig, DatasetLoader
from llm_finetuning import create_sample_instruction_dataset

path = create_sample_instruction_dataset(
    output_path="./outputs/raw_instructions.jsonl", num_samples=60
)
raw = DatasetLoader(DataConfig()).load_from_jsonl(path)

# 人为掺入脏数据，观察清洗效果
raw.append({"instruction": "正常问题", "input": "", "output": "正常回答"})
raw.append({"instruction": "缺字段样本", "input": ""})              # 缺 output
raw.append({"instruction": "空输出", "input": "", "output": "   "})  # 空白输出
raw.append({"instruction": "太短", "input": "", "output": "好"})     # 过短
raw.append(dict(raw[0]))                                            # 完全重复
raw.append({**raw[1], "output": raw[1]["output"] + "。"})            # 近似重复

print(f"原始样本数: {len(raw)}")

In [ ]:
cleaner = DataCleaner(text_field="output", min_length=5, max_length=1024, fuzzy_dedup=True)
clean, report = cleaner.clean(raw)

print(f"清洗后: {len(clean)}")
print(report.summary())

## 2. 长度分布

`length_stats` 给出均值与分位数。用它决定 `max_seq_length`：
取 **P95 附近**能覆盖绝大多数样本，又不至于为了极少数长尾浪费显存 ——
直接用 `max` 往往会被一两条异常样本带偏。

In [ ]:
from llm_finetuning import length_stats

stats = length_stats(clean, text_field="output")
for k, v in stats.items():
    print(f"  {k:<6} {v:.1f}" if isinstance(v, float) else f"  {k:<6} {v}")

print(f"\n建议 max_seq_length ≈ {int(stats['p95'] * 1.2)}（按 P95 上浮 20%）")

## 3. 可视化（可选）

有 `matplotlib` 时画个直方图，肉眼确认长尾有多长。

In [ ]:
try:
    import matplotlib.pyplot as plt

    lengths = [len(str(s.get("output", ""))) for s in clean]
    plt.figure(figsize=(7, 3))
    plt.hist(lengths, bins=20, color="#4C78A8")
    plt.xlabel("output 长度（字符）")
    plt.ylabel("样本数")
    plt.title("清洗后长度分布")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("未安装 matplotlib，跳过绘图")

## 4. 划分与落盘

`DatasetSplitter.split_train_test` 按 `test_size` 比例切分。

关键点：**先清洗去重、再划分**。顺序反了的话，同一条样本的副本会同时落进
训练集和测试集，指标会虚高。

In [ ]:
import json
from pathlib import Path

from llm_finetuning import DatasetSplitter

train, test = DatasetSplitter.split_train_test(clean, test_size=0.2, seed=42)
print(f"train={len(train)} test={len(test)}")

out_dir = Path("./outputs/data")
out_dir.mkdir(parents=True, exist_ok=True)
for name, rows in (("train", train), ("test", test)):
    fp = out_dir / f"{name}.jsonl"
    with fp.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"已写入 {fp}")